# 从时域到频域的“透视眼”：NR 工程师的 FFT 与加窗实战指南

**作者：** [余东海]  
**领域：** 5G NR 基带处理 / 信号处理  

## 一、导言
    从生活中感受傅里叶变换：
    1. 乐谱：了解简谱的人就知道，音乐是时域的，但是乐谱是频域的记录。
    2. 光谱：
    
> 什么叫频域
    时域概念我们是想对比较容易理解的


在 5G NR (New Radio) 的世界里，信号在空气中是杂乱无章的电磁波。作为软件工程师，我们最核心的任务之一就是利用 **FFT (快速傅里叶变换)** 将这些时域采样点还原为频域的子载波。
但在实际工程中，信号总是被“截断”处理的。本文将带你通过 Python 实验，直观感受 **FFT 变换**、**频谱泄露**、**加窗 (Windowing)** 以及它们对 **16-QAM 星座图** 的真实影响。

## 1. 数学基石：DFT 与 FFT
离散傅里叶变换 (DFT) 的定义如下：
$$X[k] = \sum_{n=0}^{N-1} x[n] e^{-j\frac{2\pi}{N}kn}$$

其中 $e^{-j\frac{2\pi}{N}kn}$ 被称为**旋转因子**。FFT 则是 DFT 的一种高效实现，将复杂度从 $O(N^2)$ 降到了 $O(N \log N)$。对于 NR 中高采样率（如 122.88 MHz）的信号处理，FFT 是实时性的保障。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Checkbox, VBox

# -------------------------
# 1. 原始信号参数
# -------------------------
fs = 500
t = np.linspace(0, 1, fs, endpoint=False)

# 构造一个复杂信号: 3Hz + 50Hz + 120Hz
frequencies = [3, 50, 120]
amplitudes = [1.0, 0.5, 0.2]
x = sum(a * np.sin(2*np.pi*f*t) for a, f in zip(amplitudes, frequencies))

# -------------------------
# 2. 傅里叶变换
# -------------------------
X = np.fft.fft(x)
freqs = np.fft.fftfreq(len(X), 1/fs)

# 只保留正频率
pos_mask = freqs >= 0
freqs_pos = freqs[pos_mask]
X_pos = X[pos_mask]

# -------------------------
# 3. 绘图函数
# -------------------------
def plot_selected_frequencies(selected_indices):
    plt.figure(figsize=(10,6))
    
    # 上方: 原始信号
    plt.subplot(2,1,1)
    plt.plot(t, x, color='black', label='Original Signal')
    plt.title('Original Signal (Time Domain)')
    plt.xlabel('Time [s]')
    plt.ylabel('Amplitude')
    plt.grid(True)
    plt.legend()
    
    # 下方: 选中频率分量重建信号
    y_reconstruct = np.zeros_like(x)
    for idx in selected_indices:
        if idx < len(X_pos):
            # 取对应的频域值做逆变换
            magnitude = np.abs(X_pos[idx])
            phase = np.angle(X_pos[idx])
            freq = freqs_pos[idx]
            y_reconstruct += magnitude * np.cos(2*np.pi*freq*t + phase)
    
    plt.subplot(2,1,2)
    plt.plot(t, y_reconstruct, color='red', label='Reconstructed from selected frequencies')
    plt.title(f'Selected Frequency Components')
    plt.xlabel('Time [s]')
    plt.ylabel('Amplitude')
    plt.grid(True)
    plt.legend()
    
    plt.tight_layout()
    plt.show()

# -------------------------
# 4. 创建滑块和复选框
# -------------------------
# 生成复选框列表
checkboxes = [Checkbox(value=False, description=f'{int(freqs_pos[i])} Hz') for i in range(len(frequencies))]

def interactive_plot(*args):
    selected = [i for i, cb in enumerate(checkboxes) if cb.value]
    plot_selected_frequencies(selected)

# 绑定复选框变化事件
for cb in checkboxes:
    cb.observe(interactive_plot, names='value')

# 初始绘图
interactive_plot()

# 显示复选框
VBox(checkboxes)